<a href="https://colab.research.google.com/github/Marie5559/das172-examen2-pineda-sally/blob/main/Examen2_AeroCargo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sección nueva

# Sección nueva

In [1]:
%%writefile aerocargo.py
import numpy as np

def validar_matrices(cargas, capacidades):
    """
    Módulo 1: Validación y Coherencia Dimensional.
    Verifica que las matrices de cargas reales y capacidades máximas
    cumplan con los requisitos operativos.
    """
    # Convertimos los datos de entrada a arreglos de NumPy
    cargas_np = np.array(cargas)
    capacidades_np = np.array(capacidades)

    # 1. Verificar que ambas matrices sean regulares (bidimensionales)
    if cargas_np.ndim != 2 or capacidades_np.ndim != 2:
        return False

    # 2. Verificar que tengan exactamente las mismas dimensiones N x M
    if cargas_np.shape != capacidades_np.shape:
        return False

    N, M = cargas_np.shape

    # 3. Verificar que N >= 2 y M >= 2
    if N < 2 or M < 2:
        return False

    # 4. Verificar que las cargas sean >= 0 y las capacidades > 0
    if np.any(cargas_np < 0) or np.any(capacidades_np <= 0):
        return False

    return True

Writing aerocargo.py


In [3]:
%%writefile -a aerocargo.py

def calcular_ocupacion(cargas, capacidades):
    """
    Módulo 2: Cálculo de Ocupación y Detección de Sobrecarga.
    Calcula el porcentaje de ocupación por celda e identifica
    las coordenadas con sobrecarga (>100%).
    """
    cargas_np = np.array(cargas)
    capacidades_np = np.array(capacidades)

    # Generar matriz de porcentajes de ocupación
    porcentajes = (cargas_np / capacidades_np) * 100.0

    # Identificar coordenadas de las celdas en sobrecarga (> 100.0%)
    filas, columnas = np.where(porcentajes > 100.0)
    coordenadas_sobrecarga = list(zip(filas.tolist(), columnas.tolist()))

    # Retornar estructura con matriz y lista de coordenadas
    return {
        "matriz_porcentajes": porcentajes,
        "coordenadas_criticas": coordenadas_sobrecarga
    }

Appending to aerocargo.py


In [5]:
%%writefile -a aerocargo.py

def evaluar_balance(cargas, tolerancia):
    """
    Módulo 3: Evaluación de Balance y Simetría.
    Calcula los pesos longitudinales y evalúa el desbalance lateral
    omitiendo el eje central si el número de columnas es impar.
    """
    cargas_np = np.array(cargas)

    # 1. Vector de peso total por cada fila longitudinal (eje 1 suma las columnas de cada fila)
    pesos_longitudinales = np.sum(cargas_np, axis=1).tolist()

    # 2. División de la matriz para el balance lateral
    M = cargas_np.shape[1]
    mitad = M // 2

    if M % 2 == 0:
        # Si es par, se divide exactamente a la mitad
        suma_izquierda = np.sum(cargas_np[:, :mitad])
        suma_derecha = np.sum(cargas_np[:, mitad:])
    else:
        # Si es impar, se omite la columna central
        suma_izquierda = np.sum(cargas_np[:, :mitad])
        suma_derecha = np.sum(cargas_np[:, mitad+1:])

    # 3. Cálculo del desbalance absoluto
    desbalance_lateral = abs(suma_izquierda - suma_derecha)

    # 4. Evaluación contra el umbral de tolerancia
    estado_balance = bool(desbalance_lateral <= tolerancia)

    return {
        "pesos_longitudinales": pesos_longitudinales,
        "desbalance_lateral": float(desbalance_lateral),
        "estado_balance": estado_balance
    }

Appending to aerocargo.py


In [ ]:
%%writefile -a aerocargo.py

def extraer_submatriz_critica(porcentajes, k, p):
    """
    Módulo 4: Extracción de Submatriz de Sobrecarga Crítica.
    Recorre submatrices contiguas de tamaño k x p para encontrar
    la que presente el mayor promedio de ocupación.
    """
    porcentajes_np = np.array(porcentajes)
    N, M = porcentajes_np.shape

    # Validar que la ventana k x p no sea más grande que el piso completo
    if k > N or p > M:
        return None

    max_promedio = -1.0
    submatriz_critica = None

    # Recorrer la matriz fila por fila y columna por columna
    for i in range(N - k + 1):
        for j in range(M - p + 1):
            # Extraer la submatriz actual usando "slicing" (rebanado) de NumPy
            submatriz_actual = porcentajes_np[i:i+k, j:j+p]

            # Calcular el promedio de ocupación de esa zona específica
            promedio_actual = np.mean(submatriz_actual)

            # Si este promedio es el más alto hasta ahora, lo guardamos
            if promedio_actual > max_promedio:
                max_promedio = promedio_actual
                submatriz_critica = submatriz_actual

    # Retornamos la submatriz extraída convirtiéndola de nuevo a lista normal
    return submatriz_critica.tolist()

In [6]:
%%writefile main.py
from aerocargo import validar_matrices, calcular_ocupacion, evaluar_balance, extraer_submatriz_critica

def main():
    print("=== AeroCargo-Matrix: Sistema de Auditoría de Carga ===")

    # 1. Definimos datos de prueba simulando el piso del avión (Matriz 4x4)
    cargas_reales = [
        [500, 450, 480, 510],
        [300, 310, 290, 305],
        [600, 700, 800, 650], # Colocamos un peso de 800 intencionalmente para forzar sobrecarga
        [200, 150, 150, 200]
    ]

    capacidades_maximas = [
        [600, 600, 600, 600],
        [400, 400, 400, 400],
        [600, 600, 600, 600],
        [300, 300, 300, 300]
    ]

    # Módulo 1: Validación
    es_valido = validar_matrices(cargas_reales, capacidades_maximas)
    print(f"\n1. Validación Dimensional: {'Aprobada ✅' if es_valido else 'Rechazada ❌'}")

    if not es_valido:
        print("Error: Las matrices no son operativamente válidas.")
        return

    # Módulo 2: Ocupación y Sobrecarga
    resultado_ocupacion = calcular_ocupacion(cargas_reales, capacidades_maximas)
    criticas = resultado_ocupacion['coordenadas_criticas']
    print(f"\n2. Coordenadas con Sobrecarga Crítica (>100%):")
    if criticas:
        print(f"   ⚠️ Alerta en celdas (Fila, Columna): {criticas}")
    else:
        print("   ✅ Ninguna celda supera el límite.")

    # Módulo 3: Balance y Simetría (Tolerancia de 300 kg para este vuelo)
    tolerancia_kg = 300.0
    resultado_balance = evaluar_balance(cargas_reales, tolerancia_kg)
    print(f"\n3. Evaluación de Simetría Lateral:")
    print(f"   Estado: {'Aprobado ✅' if resultado_balance['estado_balance'] else 'Desbalanceado ⚠️'}")
    print(f"   Desbalance absoluto: {resultado_balance['desbalance_lateral']} kg")
    print(f"   Vector longitudinal de masa: {resultado_balance['pesos_longitudinales']}")

    # Módulo 4: Submatriz Crítica (Evaluamos una ventana de 2x2 compartimientos)
    matriz_porcentajes = resultado_ocupacion['matriz_porcentajes']
    submatriz = extraer_submatriz_critica(matriz_porcentajes, 2, 2)
    print(f"\n4. Extracción de Submatriz Crítica (Ventana 2x2):")
    for fila in submatriz:
        print(f"   {[round(val, 1) for val in fila]} %")

if __name__ == '__main__':
    main()

Writing main.py


In [7]:
!python main.py

Traceback (most recent call last):
  File "/content/main.py", line 1, in <module>
    from aerocargo import validar_matrices, calcular_ocupacion, evaluar_balance, extraer_submatriz_critica
ImportError: cannot import name 'extraer_submatriz_critica' from 'aerocargo' (/content/aerocargo.py)


In [8]:
%%writefile aerocargo.py
import numpy as np

def validar_matrices(cargas, capacidades):
    cargas_np = np.array(cargas)
    capacidades_np = np.array(capacidades)

    if cargas_np.ndim != 2 or capacidades_np.ndim != 2:
        return False
    if cargas_np.shape != capacidades_np.shape:
        return False

    N, M = cargas_np.shape
    if N < 2 or M < 2:
        return False
    if np.any(cargas_np < 0) or np.any(capacidades_np <= 0):
        return False

    return True

def calcular_ocupacion(cargas, capacidades):
    cargas_np = np.array(cargas)
    capacidades_np = np.array(capacidades)

    porcentajes = (cargas_np / capacidades_np) * 100.0
    filas, columnas = np.where(porcentajes > 100.0)
    coordenadas_sobrecarga = list(zip(filas.tolist(), columnas.tolist()))

    return {
        "matriz_porcentajes": porcentajes,
        "coordenadas_criticas": coordenadas_sobrecarga
    }

def evaluar_balance(cargas, tolerancia):
    cargas_np = np.array(cargas)
    pesos_longitudinales = np.sum(cargas_np, axis=1).tolist()

    M = cargas_np.shape[1]
    mitad = M // 2

    if M % 2 == 0:
        suma_izquierda = np.sum(cargas_np[:, :mitad])
        suma_derecha = np.sum(cargas_np[:, mitad:])
    else:
        suma_izquierda = np.sum(cargas_np[:, :mitad])
        suma_derecha = np.sum(cargas_np[:, mitad+1:])

    desbalance_lateral = abs(suma_izquierda - suma_derecha)
    estado_balance = bool(desbalance_lateral <= tolerancia)

    return {
        "pesos_longitudinales": pesos_longitudinales,
        "desbalance_lateral": float(desbalance_lateral),
        "estado_balance": estado_balance
    }

def extraer_submatriz_critica(porcentajes, k, p):
    porcentajes_np = np.array(porcentajes)
    N, M = porcentajes_np.shape

    if k > N or p > M:
        return None

    max_promedio = -1.0
    submatriz_critica = None

    for i in range(N - k + 1):
        for j in range(M - p + 1):
            submatriz_actual = porcentajes_np[i:i+k, j:j+p]
            promedio_actual = np.mean(submatriz_actual)

            if promedio_actual > max_promedio:
                max_promedio = promedio_actual
                submatriz_critica = submatriz_actual

    return submatriz_critica.tolist()

Overwriting aerocargo.py


In [9]:
!python main.py

=== AeroCargo-Matrix: Sistema de Auditoría de Carga ===

1. Validación Dimensional: Aprobada ✅

2. Coordenadas con Sobrecarga Crítica (>100%):
   ⚠️ Alerta en celdas (Fila, Columna): [(2, 1), (2, 2), (2, 3)]

3. Evaluación de Simetría Lateral:
   Estado: Aprobado ✅
   Desbalance absoluto: 175.0 kg
   Vector longitudinal de masa: [1940, 1205, 2750, 700]

4. Extracción de Submatriz Crítica (Ventana 2x2):
   [77.5, 72.5] %
   [116.7, 133.3] %


In [10]:
%%writefile test_aerocargo.py
import unittest
from aerocargo import validar_matrices, evaluar_balance

class TestAeroCargo(unittest.TestCase):

    def test_validacion_correcta(self):
        # Caso típico: matrices regulares 2x2 con valores correctos
        cargas = [[100, 200], [300, 400]]
        capacidades = [[500, 500], [500, 500]]
        self.assertTrue(validar_matrices(cargas, capacidades))

    def test_validacion_pesos_negativos(self):
        # Caso límite: simulamos un error de digitación con un peso negativo (-200)
        cargas = [[100, -200], [300, 400]]
        capacidades = [[500, 500], [500, 500]]
        self.assertFalse(validar_matrices(cargas, capacidades))

    def test_evaluar_balance_impar(self):
        # Caso límite: matriz con 3 columnas (impar). El código debe ignorar la columna central.
        cargas = [[100, 800, 100], [200, 900, 200]]
        resultado = evaluar_balance(cargas, tolerancia=50.0)
        self.assertTrue(resultado['estado_balance'])
        self.assertEqual(resultado['desbalance_lateral'], 0.0)

if __name__ == '__main__':
    unittest.main()

Writing test_aerocargo.py


In [11]:
!python -m unittest test_aerocargo.py

...
----------------------------------------------------------------------
Ran 3 tests in 0.001s

OK
